# PHA 快速上手

用几行代码体验个人健康 Agent（PHA）。

## PHA 是什么？

PHA 结合三个方面分析健康数据并给出个性化见解。

- **数据分析**：识别趋势和规律，与参考人群比较。
- **医学解读**：解释各项指标的含义。
- **健康指导**：给出可执行的建议。

## 环境要求

```bash
pip install google-genai pandas openai anthropic
# 可选：安装 OneTwo，启用领域专家 Agent 的完整功能
pip install git+https://github.com/google-deepmind/onetwo
```

## 快速配置

In [ ]:
import os
import sys

# --- Configuration ---
# 在此设定您的 API 密钥和提供者
API_KEY = "API_KEY"
# 提供方选项："基因","openai","人类"
PROVIDER = "PROVIDER"
# 可选：Tavily API 键 领域专家 Agent (搜索).
TAVILY_API_KEY = "TAVILY_API_KEY"

In [ ]:
import pandas as pd
import glob
from pha.agents import (
    DataScienceAgent,
    DomainExpertAgent,
    HealthCoachAgent,
    MultiAgentOrchestrator,
    is_react_available,
)

# 装入您的健康数据(在此使用样本数据)
summary_df = pd.read_csv('../data/sample/summary.csv')
activities_df = pd.read_csv('../data/sample/activities.csv')
profile_df = pd.read_csv('../data/sample/profile.csv')
population_df = pd.read_csv('../data/sample/population_percentiles.csv')

print(f"已加载 {len(summary_df)} 天的健康数据")
print(f"ReAct 可用： {is_react_available()}")


In [ ]:
# 初始化 数据科学 Agent。
ds_agent = DataScienceAgent()
ds_agent.configure(api_key=API_KEY, provider=PROVIDER)  # 明确的配置
ds_agent.load_dataframes({
    'summary': summary_df,
    'activities': activities_df,
    'profile': profile_df,
    'population': population_df,
})
print("✓ 数据科学 Agent 就绪")

# 初始化 领域专家 Agent。
de_agent = None
if is_react_available():
    de_agent = DomainExpertAgent(
        search_backend='tavily',
        tavily_api_key=TAVILY_API_KEY,
    )
    exemplar_files = glob.glob('../few_shots/*.ipynb')
    de_agent.get_agent(
        api_key=API_KEY,
        provider=PROVIDER,
        exemplar_files=exemplar_files,
    )
    print(f"✓ 领域专家 Agent 就绪（{len(exemplar_files)} 个示例）")
else:
    print("⚠ 领域专家 Agent 不可用，安装 OneTwo 可启用完整功能")

# 启动卫生教练
coach = HealthCoachAgent(simple_mode=True)
coach.configure(api_key=API_KEY, provider=PROVIDER)  # 明确的配置
print("✓ 健康教练就绪")

# 创建所有代理的 编排器
pha = MultiAgentOrchestrator()
pha.configure(api_key=API_KEY, provider=PROVIDER)  # 明确的配置
pha.set_agents(
    data_science_agent=ds_agent,
    domain_expert_agent=de_agent,
    health_coach_agent=coach,
)

print("\n✓ PHA 就绪")


## 提出健康问题

In [ ]:
# 问一个健康问题
response = pha.respond("我最近睡得怎么样？")
print(response)

In [ ]:
# 询问后续问题
response = pha.respond("我的活动水平呢？")
print(response)

In [ ]:
# 获得个性化建议
response = pha.respond("我应该优先改善哪些方面？")
print(response)

## 单独调用各个 Agent

也可以独立使用每个 Agent。

In [ ]:
# 数据科学 Agent - 用于数据分析
analysis = ds_agent.query("我每天平均走多少步？")
print("数据科学 Agent：")
print(analysis)

In [ ]:
# 健康教练 -- -- 谈话指导
coach.reset_conversation()
response = coach.respond("我想让自己更有精神。")
print("Health 健康教练：")
print(response)

In [ ]:
# 领域专家 -- -- 健康解释和背景
if de_agent:
    response = de_agent.call_agent("糖化血红蛋白（HbA1c）为 5.8%，这对我的健康意味着什么？")
    print("领域专家 Agent：")
    print(response)
else:
    print("领域专家 Agent 不可用，需要 OneTwo")


## 后续阅读

- `02_data_science_agent.ipynb` 展示详细的数据分析。
- `03_domain_expert_agent.ipynb` 展示如何解读健康指标。
- `04_health_coach_agent.ipynb` 展示对话式健康指导。
- `05_full_pipeline.ipynb` 展示完整的编排流程。

## 加载自己的数据

```python
# 读取自己的健康数据
my_summary = pd.read_csv('path/to/your/summary.csv')
my_activities = pd.read_csv('path/to/your/activities.csv')

ds_agent.load_dataframes({
    'summary': my_summary,
    'activities': my_activities,
})
```

所需字段包括以下几项。

- **summary**：`datetime`、`steps`、`sleep_minutes`、`resting_heart_rate`、`heart_rate_variability` 等。
- **activities**：`start_time`、`activity_name`、`duration`、`calories`、`steps` 等。